## Cell 1 — Setup

In [12]:
import os, pathlib, json, shutil
import numpy as np
from functools import partial
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp
from qonnx.transformation.general import GiveUniqueNodeNames, RemoveUnusedTensors
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.core.datatype import DataType
from finn.util.basic import pynq_part_map
from finn.util.visualization import showInNetron
import finn.transformation.streamline.absorb as absorb

# ⚠️ Updated to our new Fashion-MNIST folder
BUILD_DIR = pathlib.Path('/home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/')
BOARD     = "Pynq-Z2"
FPGA_PART = pynq_part_map[BOARD]
CLK_NS    = 10.0

# ⚠️ Updated all filenames for the new MobileNet model
FINN_TIDY     = str(BUILD_DIR / 'fashion_mobilenet_finn_tidy.onnx')
STREAMLINED   = str(BUILD_DIR / 'fashion_mobilenet_streamlined.onnx')
DATAFLOW_PAR  = str(BUILD_DIR / 'fashion_mobilenet_dataflow_parent.onnx')
DATAFLOW_MDL  = str(BUILD_DIR / 'fashion_mobilenet_dataflow_model.onnx')
FOLDED        = str(BUILD_DIR / 'fashion_mobilenet_folded.onnx')
SYNTH_MODEL   = str(BUILD_DIR / 'fashion_mobilenet_hlssynth.onnx')
ZYNQ_MODEL    = str(BUILD_DIR / 'fashion_mobilenet_zynq.onnx')

print(f"Board : {BOARD}  |  FPGA : {FPGA_PART}  |  Clock : {1000/CLK_NS:.0f} MHz")

Board : Pynq-Z2  |  FPGA : xc7z020clg400-1  |  Clock : 100 MHz


## Cell 2 — Delete Stale Files

In [13]:
stale = [
    'fashion_mobilenet_streamlined.onnx', 'fashion_mobilenet_dataflow_parent.onnx',
    'fashion_mobilenet_dataflow_model.onnx', 'fashion_mobilenet_folded.onnx',
    'fashion_mobilenet_hlssynth.onnx', 'fashion_mobilenet_zynq.onnx'
]

for fname in stale:
    f = BUILD_DIR / fname
    if f.exists(): 
        f.unlink()
        print(f"Deleted: {fname}")

for folder in os.listdir(str(BUILD_DIR)):
    full = os.path.join(str(BUILD_DIR), folder)
    if os.path.isdir(full) and any(x in folder for x in ['code_gen', 'vivado', 'ipgen']):
        shutil.rmtree(full)
        print(f"Deleted dir: {folder}")
        
print("✅ Clean")

✅ Clean


## Cell 3 — Streamlining

With MaxPool moved AFTER QuantReLU, BN (Mul+Add) now sits directly
before MultiThreshold and Streamline absorbs it cleanly.


In [14]:
from finn.transformation.streamline import Streamline
from finn.transformation.streamline.reorder import MoveScalarLinearPastInvariants
from finn.transformation.streamline.absorb import AbsorbTransposeIntoMultiThreshold
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors
import pathlib
from qonnx.core.modelwrapper import ModelWrapper

# --- PATH SETUP ---
BUILD_DIR = pathlib.Path('.') # Adjust if your files are in a specific folder
FINN_TIDY = str(BUILD_DIR / 'fashion_cnn_finn_tidy.onnx')
STREAMLINED = str(BUILD_DIR / 'fashion_cnn_streamlined.onnx')

print("Loading tidy model...")
model = ModelWrapper(FINN_TIDY)

print("Streamlining...")
model = model.transform(MoveScalarLinearPastInvariants())
model = model.transform(Streamline())
model = model.transform(LowerConvsToMatMul())
model = model.transform(AbsorbTransposeIntoMultiThreshold())
model = model.transform(Streamline())
model = model.transform(FoldConstants())

# Clean up types and layouts
model = model.transform(InferDataTypes())
model = model.transform(InferDataLayouts())
model = model.transform(RemoveUnusedTensors())

model.save(STREAMLINED)
print(f"✅ Streamlined successfully! Nodes = {len(model.graph.node)}")

Loading tidy model...
Streamlining...


Task was destroyed but it is pending!
task: <Task pending name='Task-121' coro=<_async_in_context.<locals>.run_in_context_pre311() done, defined at /usr/local/lib/python3.10/dist-packages/ipykernel/utils.py:76> wait_for=<Task pending name='Task-122' coro=<_async_in_context.<locals>.preserve_context() running at /usr/local/lib/python3.10/dist-packages/ipykernel/utils.py:68> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /usr/local/lib/python3.10/dist-packages/zmq/eventloop/zmqstream.py:563]>
/usr/local/lib/python3.10/dist-packages/toposort.py:81: RuntimeWarning: coroutine '_async_in_context.<locals>.preserve_context' was never awaited
  item: (dep - ordered) for item, dep in data.items() if item not in ordered
Task was destroyed but it is pending!
task: <Task pending name='Task-122' coro=<_async_in_context.<locals>.preserve_context() running at /usr/local/lib/python3.10/dist-packages/ipykernel/utils.py:68> cb=[Task.task_wakeup()]>


✅ Streamlined successfully! Nodes = 28


## Cell 4 — HW Layer Conversion

In [15]:
import os, pathlib
import onnx.helper as oh
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.general import GiveUniqueNodeNames
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.custom_op.registry import getCustomOp

import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers
from finn.transformation.fpgadataflow.create_dataflow_partition import CreateDataflowPartition
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten
# 🌟 THE REAL FINN FUNCTION: This flips the memory layout so the FPGA can absorb it
from finn.transformation.streamline.reorder import MakeMaxPoolNHWC
import finn.transformation.streamline.absorb as absorb
from finn.util.basic import pynq_part_map

# --- BOARD CONFIG ---
BOARD      = "Pynq-Z2"
FPGA_PART  = pynq_part_map[BOARD]

BUILD_DIR = pathlib.Path('.')
STREAMLINED = str(BUILD_DIR / 'fashion_cnn_streamlined.onnx')
DATAFLOW_PAR = str(BUILD_DIR / 'fashion_cnn_dataflow_parent.onnx')
DATAFLOW_MDL = str(BUILD_DIR / 'fashion_cnn_dataflow_model.onnx')

print("Loading streamlined model...")
model = ModelWrapper(STREAMLINED)

# ==========================================
# STEP 1 — Fix Layouts & Infer HW
# ==========================================
print("Step 1: Reordering MaxPool layouts and Inferring HW Layers...")
# Flip the MaxPool to NHWC so the hardware compiler understands it
model = model.transform(MakeMaxPoolNHWC())
model = model.transform(absorb.AbsorbConsecutiveTransposes())

model = model.transform(to_hw.InferQuantizedMatrixVectorActivation())
model = model.transform(to_hw.InferThresholdingLayer())
model = model.transform(to_hw.InferConvInpGen())
# Now that the layout is NHWC, we can safely convert it to hardware
model = model.transform(to_hw.InferStreamingMaxPool())  
model = model.transform(to_hw.InferLabelSelectLayer()) 

# ==========================================
# STEP 2 — Flattening Cleanup
# ==========================================
print("Step 2: Cleaning up Flattening nodes...")
model = model.transform(RemoveCNVtoFCFlatten())
model = model.transform(absorb.AbsorbConsecutiveTransposes())
model = model.transform(absorb.AbsorbTransposeIntoMultiThreshold())

# ==========================================
# STEP 3 — The Failsafe (Mul Cleanup)
# ==========================================
print("Step 3: Removing final software blockers...")
nodes = list(model.graph.node)
out2node = {n.output[0]: n for n in nodes if n.output}
for topk in [n for n in nodes if n.op_type == 'TopK']:
    mul = out2node.get(topk.input[0])
    if mul and mul.op_type == 'Mul':
        dyn_inp = mul.input[0] if model.get_initializer(mul.input[0]) is None else mul.input[1]
        topk.input[0] = dyn_inp
        model.graph.node.remove(mul)
        print("   🔧 Removed: Leftover Mul node before TopK")

# ==========================================
# STEP 4 — Specialize & Infer
# ==========================================
print("Step 4: Specializing layers for Pynq-Z2...")
model = model.transform(GiveUniqueNodeNames())
model = model.transform(SpecializeLayers(FPGA_PART))
model = model.transform(InferShapes())
model = model.transform(InferDataTypes())
model = model.transform(InferDataLayouts())
model = model.transform(GiveUniqueNodeNames())

# ==========================================
# STEP 5 — Check Non-HW Nodes
# ==========================================
non_hw = [n.op_type for n in model.graph.node if "fpgadataflow" not in n.domain]
print(f"Nodes executing safely on ARM CPU: {set(non_hw)}")

# ==========================================
# STEP 6 — Partitioning
# ==========================================
print("Step 6: Partitioning Dataflow...")
parent_model = model.transform(CreateDataflowPartition())
parent_model.save(DATAFLOW_PAR)

sdp_nodes = parent_model.get_nodes_by_op_type("StreamingDataflowPartition")

if len(sdp_nodes) > 0:
    sdp_node = sdp_nodes[0]
    df_file  = getCustomOp(sdp_node).get_nodeattr("model")
    df_model = ModelWrapper(df_file)
    df_model = df_model.transform(GiveUniqueNodeNames())
    df_model.save(DATAFLOW_MDL)
    print("\n✅ DATAFLOW MODEL SUCCESSFULLY PARTITIONED AND SAVED!")
else:
    print("\n❌ Failed to create StreamingDataflowPartition.")

Loading streamlined model...
Step 1: Reordering MaxPool layouts and Inferring HW Layers...
Step 2: Cleaning up Flattening nodes...
Step 3: Removing final software blockers...
   🔧 Removed: Leftover Mul node before TopK
Step 4: Specializing layers for Pynq-Z2...
Nodes executing safely on ARM CPU: {'TopK', 'Transpose'}
Step 6: Partitioning Dataflow...

✅ DATAFLOW MODEL SUCCESSFULLY PARTITIONED AND SAVED!


## Cell 5 — Check Layer Dimensions

In [16]:
df_model   = ModelWrapper(DATAFLOW_MDL)
mvau_nodes = df_model.get_nodes_by_op_type("MVAU_hls") + df_model.get_nodes_by_op_type("MVAU_rtl")
swg_nodes  = df_model.get_nodes_by_op_type("ConvolutionInputGenerator_rtl")
pool_nodes = df_model.get_nodes_by_op_type("StreamingMaxPool_hls")
print(f"MVAU:{len(mvau_nodes)}  SWG:{len(swg_nodes)}  Pool:{len(pool_nodes)}")
print("\nLayer dimensions:")
for node in mvau_nodes:
    inst = getCustomOp(node)
    mw   = inst.get_nodeattr("MW")
    mh   = inst.get_nodeattr("MH")
    print(f"  {node.name:35s}  MW={mw:6d}  MH={mh:5d}")

MVAU:5  SWG:4  Pool:4

Layer dimensions:
  MVAU_hls_0                           MW=     9  MH=   64
  MVAU_hls_1                           MW=   576  MH=   64
  MVAU_hls_2                           MW=   576  MH=  128
  MVAU_hls_3                           MW=  1152  MH=  128
  MVAU_hls_4                           MW=   512  MH=   10


## Cell 6 — Folding

New architecture with MaxPool after QuantReLU:

| Layer | MW | MH | Notes |
|-------|----|----|-------|
| Conv1 | 27 | 64 | W8, pool after |
| Conv2 | 576 | 64 | W4A4, pool after |
| Conv3 | 576 | 128 | W4A4, pool after |
| Conv4 | 1152 | 128 | W4A4 |
| FC5 | 2048 | 256 | W4A4 (128×4×4) |
| FC6 | 256 | 10 | W8 head |


In [17]:
import numpy as np
import pathlib
import math
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp
from qonnx.transformation.general import GiveUniqueNodeNames

BUILD_DIR = pathlib.Path('.')
DATAFLOW_MDL = str(BUILD_DIR / 'fashion_cnn_dataflow_model.onnx')
FOLDED = str(BUILD_DIR / 'fashion_cnn_folded.onnx')

df_model = ModelWrapper(DATAFLOW_MDL)

mvau_nodes = df_model.get_nodes_by_op_type("MVAU_hls") + df_model.get_nodes_by_op_type("MVAU_rtl")
swg_nodes = df_model.get_nodes_by_op_type("ConvolutionInputGenerator_rtl") + df_model.get_nodes_by_op_type("ConvolutionInputGenerator_hls")

print("🎛️ Applying Safe All-BRAM Folding...")

# Configure Compute Nodes
for node in mvau_nodes:
    inst = getCustomOp(node)
    mw = inst.get_nodeattr("MW")
    simd = math.ceil(mw / 1024) if mw > 1024 else 1
    while mw % simd != 0: simd += 1
            
    inst.set_nodeattr("PE", 1)
    inst.set_nodeattr("SIMD", simd)
    inst.set_nodeattr("ram_style", "block") 

# Configure Memory Nodes
for swg in swg_nodes:
    inst = getCustomOp(swg)
    inst.set_nodeattr("SIMD", 1)
    inst.set_nodeattr("ram_style", "block") 

df_model = df_model.transform(GiveUniqueNodeNames())
df_model.save(FOLDED)
print(f"✅ Folding complete -> {FOLDED}")

🎛️ Applying Safe All-BRAM Folding...
✅ Folding complete -> fashion_cnn_folded.onnx


## Cell 7 — Resource Estimates

In [27]:
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp

df_model = ModelWrapper(FOLDED)
print(f"{'Node Name':<40} | {'MH':>5} | {'PE':>5} | {'Valid?'}")
print("-" * 65)

for n in df_model.graph.node:
    try:
        inst = getCustomOp(n)
        # Check MVAU (Matrix Vector Activation Unit) layers
        if n.op_type in ["MVAU_hls", "MVAU_rtl"]:
            mh = inst.get_nodeattr("MH")
            pe = inst.get_nodeattr("PE")
            is_valid = (mh % pe == 0)
            status = "✅" if is_valid else "❌ BROKEN"
            print(f"{n.name:<40} | {mh:>5} | {pe:>5} | {status}")
            
        # Check Thresholding layers
        elif "Thresholding" in n.op_type:
            # Thresholding MH is usually the number of channels
            num_ch = inst.get_nodeattr("NumChannels")
            pe = inst.get_nodeattr("PE")
            is_valid = (num_ch % pe == 0)
            status = "✅" if is_valid else "❌ BROKEN"
            print(f"{n.name:<40} | {num_ch:>5} | {pe:>5} | {status}")
            
    except:
        continue

Node Name                                |    MH |    PE | Valid?
-----------------------------------------------------------------
MVAU_hls_0                               |    64 |     1 | ✅
MVAU_hls_1                               |    64 |     1 | ✅
MVAU_hls_2                               |   128 |     1 | ✅
MVAU_hls_3                               |   128 |     1 | ✅
MVAU_hls_4                               |    10 |     1 | ✅


In [19]:
from finn.analysis.fpgadataflow.exp_cycles_per_layer import exp_cycles_per_layer
from finn.analysis.fpgadataflow.res_estimation import res_estimation
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp
from functools import partial

print("🕵️ Analyzing FOLDED model for PYNQ-Z2 resources...")
df_model = ModelWrapper(FOLDED)

# 1. Check for Dim/PE mismatches before running full analysis
print("Checking Integer Divisibility...")
for n in df_model.graph.node:
    try:
        inst = getCustomOp(n)
        if n.op_type in ["MVAU_hls", "MVAU_rtl"]:
            mh = inst.get_nodeattr("MH")
            mw = inst.get_nodeattr("MW")
            pe = inst.get_nodeattr("PE")
            simd = inst.get_nodeattr("SIMD")
            assert mh % pe == 0, f"❌ {n.name}: MH({mh}) not div by PE({pe})"
            assert mw % simd == 0, f"❌ {n.name}: MW({mw}) not div by SIMD({simd})"
    except Exception as e:
        if "not div by" in str(e):
            print(e)

# 2. Run the actual FINN Analysis
try:
    res_func = partial(res_estimation, fpgapart=FPGA_PART)
    cycles = df_model.analysis(exp_cycles_per_layer)
    res    = df_model.analysis(res_func)

    total_luts = sum([res[k].get('LUT', 0) for k in res])
    total_bram = sum([res[k].get('BRAM_18K', 0) for k in res])

    print("-" * 50)
    print(f"Total LUTs  : {int(total_luts):,} / 53,200")
    print(f"Total BRAMs : {int(total_bram):,} / 140")
    print("-" * 50)
    
    if total_luts < 53200 and total_bram < 140:
        print("✅ READY FOR VIVADO")
    else:
        print("⚠️ OVER BUDGET")

except Exception as e:
    print(f"\n💥 Analysis Failed: {str(e)}")

🕵️ Analyzing FOLDED model for PYNQ-Z2 resources...
Checking Integer Divisibility...
--------------------------------------------------
Total LUTs  : 3,021 / 53,200
Total BRAMs : 41 / 140
--------------------------------------------------
✅ READY FOR VIVADO


/home/yeager/finn/src/finn/custom_op/fpgadataflow/streamingmaxpool.py:139: UserWarning: Estimated latency for layer StreamingMaxPool_hls_0 can be lower than
             actual latency!
  warnings.warn(
/home/yeager/finn/src/finn/custom_op/fpgadataflow/streamingmaxpool.py:139: UserWarning: Estimated latency for layer StreamingMaxPool_hls_1 can be lower than
             actual latency!
  warnings.warn(
/home/yeager/finn/src/finn/custom_op/fpgadataflow/streamingmaxpool.py:139: UserWarning: Estimated latency for layer StreamingMaxPool_hls_2 can be lower than
             actual latency!
  warnings.warn(
/home/yeager/finn/src/finn/custom_op/fpgadataflow/streamingmaxpool.py:139: UserWarning: Estimated latency for layer StreamingMaxPool_hls_3 can be lower than
             actual latency!
  warnings.warn(


## Cell 8 — HLS Synthesis

In [23]:
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
import os
import pathlib

BUILD_DIR = pathlib.Path('/home/yeager/finn/notebooks/claude_builds/mnist_1_fashion')
model_file = str(BUILD_DIR / "fashion_cnn_folded.onnx")

vivado_output_dir = str(BUILD_DIR / "vivado_output")
os.environ["FINN_BUILD_DIR"] = vivado_output_dir

cfg = build_cfg.DataflowBuildConfig(
    output_dir          = vivado_output_dir,
    
    # 🌟 THE FIX: Drop target_fps so FINN doesn't override our PE=1 setting!
    target_fps          = 10, 
    
    synth_clk_period_ns = 10.0,
    board               = "Pynq-Z2",
    fpga_part           = "xc7z020clg400-1", 
    generate_outputs    = [
        build_cfg.DataflowOutputType.BITFILE,
        build_cfg.DataflowOutputType.PYNQ_DRIVER,
        build_cfg.DataflowOutputType.DEPLOYMENT_PACKAGE,
    ]
)

print(f"🚀 Sending {model_file} to FINN Builder (Low FPS Target)...")
build.build_dataflow_cfg(model_file, cfg)
print(f"✅ FINN Build Complete! Files saved to: {vivado_output_dir}")

🚀 Sending /home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/fashion_cnn_folded.onnx to FINN Builder (Low FPS Target)...
Building dataflow accelerator from /home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/fashion_cnn_folded.onnx
Intermediate outputs will be generated in /home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/vivado_output
Final outputs will be generated in /home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/vivado_output
Build log is at /home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/vivado_output/build_dataflow.log
Running step: step_qonnx_to_finn [1/19]
Running step: step_tidy_up [2/19]
Running step: step_streamline [3/19]
Running step: step_convert_to_hw [4/19]
Running step: step_create_dataflow_partition [5/19]
Running step: step_specialize_layers [6/19]
Running step: step_target_fps_parallelization [7/19]
Running step: step_apply_folding_config [8/19]
Running step: step_minimize_bit_width [9/19]
Running step: step_generate_estimat

Traceback (most recent call last):
  File "/home/yeager/finn/src/finn/builder/build_dataflow.py", line 158, in build_dataflow_cfg
    model = transform_step(model, cfg)
  File "/home/yeager/finn/src/finn/builder/build_dataflow_steps.py", line 515, in step_hw_codegen
    model = model.transform(PrepareIP(cfg._resolve_fpga_part(), cfg._resolve_hls_clk_period()))
  File "/home/yeager/finn/deps/qonnx/src/qonnx/core/modelwrapper.py", line 140, in transform
    (transformed_model, model_was_changed) = transformation.apply(transformed_model)
  File "/home/yeager/finn/src/finn/transformation/fpgadataflow/prepare_ip.py", line 94, in apply
    _codegen_single_node(node, model, self.fpgapart, self.clk)
  File "/home/yeager/finn/src/finn/transformation/fpgadataflow/prepare_ip.py", line 54, in _codegen_single_node
    inst.code_generation_ipgen(model, fpgapart, clk)
  File "/home/yeager/finn/src/finn/custom_op/fpgadataflow/hlsbackend.py", line 120, in code_generation_ipgen
    self.defines("ipgen")

ipdb>  q


Build failed
✅ FINN Build Complete! Files saved to: /home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/vivado_output


In [25]:
import shutil, glob, os, pathlib
import torch
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.general import GiveUniqueNodeNames
from finn.transformation.fpgadataflow.prepare_ip import PrepareIP
from finn.transformation.fpgadataflow.hlssynth_ip import HLSSynthIP
from finn.transformation.fpgadataflow.insert_dwc import InsertDWC
from finn.transformation.fpgadataflow.insert_fifo import InsertFIFO
from finn.transformation.fpgadataflow.make_zynq_proj import ZynqBuild

# --- 1. CONFIGURATION & PATHS ---
BUILD_DIR   = pathlib.Path('/home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/')
BOARD       = "Pynq-Z2"
FPGA_PART   = "xc7z020clg400-1"
CLK_NS      = 20.0 # 50 MHz (This slower clock makes Vivado routing much easier!)

# 🌟 THE FIX: Pointing to your new 4-layer CNN files!
FOLDED      = str(BUILD_DIR / "fashion_cnn_folded.onnx")
SYNTH_MODEL = str(BUILD_DIR / "fashion_cnn_hlssynth.onnx")
ZYNQ_MODEL  = str(BUILD_DIR / "fashion_cnn_zynq.onnx")

# --- 2. HLS SYNTHESIS (10-20 mins) ---
print(f"📂 Loading folded model from: {FOLDED}")
df_model = ModelWrapper(FOLDED)
df_model = df_model.transform(GiveUniqueNodeNames())

print("⚙️ PrepareIP …")
df_model = df_model.transform(PrepareIP(FPGA_PART, CLK_NS))

print("⚙️ HLSSynthIP … (Expect 10–20 min)")
df_model = df_model.transform(HLSSynthIP())
df_model.save(SYNTH_MODEL)
print(f"✅ HLS done → {SYNTH_MODEL}")

# --- 3. CLEAR STALE CACHE ---
print("\n🧹 Clearing stale HLS IP cache for a clean build...")
stale_dirs_removed = 0
patterns = [
    str(BUILD_DIR / "code_gen_ipgen_*"),
    "/tmp/finn_dev_*/code_gen_ipgen_*",
    "/tmp/finn_dev_*/vivado_zynq_proj_*"
]
for pattern in patterns:
    for d in glob.glob(pattern, recursive=True):
        if os.path.isdir(d):
            shutil.rmtree(d)
            stale_dirs_removed += 1
print(f"✨ Cleared {stale_dirs_removed} stale directories.")

# --- 4. DATA HIGHWAY OPTIMIZATION ---
synth_model = ModelWrapper(SYNTH_MODEL)
print("\n🛠️ Fixing data highway widths (Inserting DWCs)...")
synth_model = synth_model.transform(InsertDWC())

print("💉 Injecting FIFOs for hardware stability...")
synth_model = synth_model.transform(InsertFIFO(True))
synth_model = synth_model.transform(GiveUniqueNodeNames())

# --- 5. THE VIVADO BUILD (30-90 mins) ---
print(f"\n🚀 Launching ZynqBuild for {BOARD}...")
print("Vivado Synthesis & Implementation started. Time for a coffee break! ☕")

synth_model = synth_model.transform(ZynqBuild(platform=BOARD, period_ns=CLK_NS))
synth_model.save(ZYNQ_MODEL)

# --- 6. LOCATE THE PRIZE ---
bitfile_path = synth_model.get_metadata_prop('bitfile')
if bitfile_path:
    print(f"\n✅ SUCCESS! Bitfile generated at: {bitfile_path}")
    hwh_path = bitfile_path.replace(".bit", ".hwh")
    print(f"📦 HWH file located at: {hwh_path}")
else:
    print("\n❌ Build finished but bitfile path not found in metadata. Check Vivado logs.")

📂 Loading folded model from: /home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/fashion_cnn_folded.onnx
⚙️ PrepareIP …
⚙️ HLSSynthIP … (Expect 10–20 min)
✅ HLS done → /home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/fashion_cnn_hlssynth.onnx

🧹 Clearing stale HLS IP cache for a clean build...
✨ Cleared 0 stale directories.

🛠️ Fixing data highway widths (Inserting DWCs)...
💉 Injecting FIFOs for hardware stability...

🚀 Launching ZynqBuild for Pynq-Z2...
Vivado Synthesis & Implementation started. Time for a coffee break! ☕


/home/yeager/finn/src/finn/transformation/fpgadataflow/floorplan.py:107: UserWarning: 55 nodes have no entry in the provided floorplan, SLR was set to -1
  warnings.warn(
/home/yeager/finn/src/finn/transformation/fpgadataflow/prepare_ip.py:56: UserWarning: Using pre-existing code for StreamingDataflowPartition_1_FMPadding_rtl_0
  warnings.warn("Using pre-existing code for %s" % node.name)
/home/yeager/finn/src/finn/transformation/fpgadataflow/prepare_ip.py:56: UserWarning: Using pre-existing code for StreamingDataflowPartition_1_ConvolutionInputGenerator_rtl_0
  warnings.warn("Using pre-existing code for %s" % node.name)
/home/yeager/finn/src/finn/transformation/fpgadataflow/prepare_ip.py:56: UserWarning: Using pre-existing code for StreamingDataflowPartition_1_MVAU_hls_0
  warnings.warn("Using pre-existing code for %s" % node.name)
/home/yeager/finn/src/finn/transformation/fpgadataflow/prepare_ip.py:56: UserWarning: Using pre-existing code for StreamingDataflowPartition_1_StreamingMax

Exception: Synthesis failed, no bitfile found. Check logs under /home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/vivado_output/vivado_zynq_proj__2wi08av

 Cell 9 — Vivado + Bitfile

In [ ]:
from finn.transformation.fpgadataflow.prepare_ip import PrepareIP
from finn.transformation.fpgadataflow.hlssynth_ip import HLSSynthIP

df_model = ModelWrapper(FOLDED)
df_model = df_model.transform(GiveUniqueNodeNames())
print("PrepareIP …")
df_model = df_model.transform(PrepareIP(FPGA_PART, CLK_NS))
print("HLSSynthIP … (10–20 min)")
df_model = df_model.transform(HLSSynthIP())
df_model.save(SYNTH_MODEL)
print(f"✅ HLS done → {SYNTH_MODEL}")
import shutil, glob, os, pathlib
import torch
from finn.transformation.fpgadataflow.insert_dwc import InsertDWC
from finn.transformation.fpgadataflow.insert_fifo import InsertFIFO
from finn.transformation.fpgadataflow.make_zynq_proj import ZynqBuild
from qonnx.transformation.general import GiveUniqueNodeNames
from qonnx.core.modelwrapper import ModelWrapper

# --- 1. CONFIGURATION & PATHS ---
# Ensure these match the folder you have been using
BUILD_DIR   = pathlib.Path('/home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/')
BOARD       = "Pynq-Z2"
CLK_NS      = 20.0 # 50 MHz

# The hand-off file from the previous HLS step
SYNTH_MODEL = str(BUILD_DIR / "fashion_mobilenet_hlssynth.onnx")
# The final output file for the Zynq project
ZYNQ_MODEL  = str(BUILD_DIR / "fashion_mobilenet_zynq.onnx")

# Load the model
print(f"📂 Loading HLS model from: {SYNTH_MODEL}")
synth_model = ModelWrapper(SYNTH_MODEL)

# --- 2. CLEAR STALE CACHE ---
# Vivado is aggressive with caching; we wipe local IP folders to force a fresh run
print("🧹 Clearing stale HLS IP cache for a clean build...")
stale_dirs_removed = 0
patterns = [
    str(BUILD_DIR / "code_gen_ipgen_*"),
    "/tmp/finn_dev_*/code_gen_ipgen_*",
    "/tmp/finn_dev_*/vivado_zynq_proj_*"
]

for pattern in patterns:
    for d in glob.glob(pattern, recursive=True):
        if os.path.isdir(d):
            shutil.rmtree(d)
            stale_dirs_removed += 1
            print(f"  Deleted: {d}")
print(f"✨ Cleared {stale_dirs_removed} stale directories.")

# --- 3. DATA HIGHWAY OPTIMIZATION ---
# These add Data Width Converters (DWC) and FIFOs for timing closure
print("\n🛠️ Fixing data highway widths (Inserting DWCs)...")
synth_model = synth_model.transform(InsertDWC())

print("💉 Injecting FIFOs for hardware stability...")
synth_model = synth_model.transform(InsertFIFO(True))
synth_model = synth_model.transform(GiveUniqueNodeNames())

# --- 4. THE VIVADO BUILD ---
# This is the 'Heavy Lifter'. Expect 30-90 minutes of processing.
print(f"\n🚀 Launching ZynqBuild for {BOARD}...")
print("Vivado Synthesis & Implementation started. Time for a coffee break! ☕")

# This transformation invokes the Vivado toolchain to create the bitstream
synth_model = synth_model.transform(ZynqBuild(platform=BOARD, period_ns=CLK_NS))
synth_model.save(ZYNQ_MODEL)

# --- 5. LOCATE THE PRIZE ---
bitfile_path = synth_model.get_metadata_prop('bitfile')
if bitfile_path:
    print(f"\n✅ SUCCESS! Bitfile generated at: {bitfile_path}")
    # Also grab the hardware handoff file (.hwh) for the PYNQ driver
    hwh_path = bitfile_path.replace(".bit", ".hwh")
    print(f"📦 HWH file located at: {hwh_path}")
else:
    print("\n❌ Build finished but bitfile path not found in metadata. Check Vivado logs.")

In [26]:
import shutil, glob, os, pathlib
import torch
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.general import GiveUniqueNodeNames
from qonnx.custom_op.registry import getCustomOp
from finn.transformation.fpgadataflow.prepare_ip import PrepareIP
from finn.transformation.fpgadataflow.hlssynth_ip import HLSSynthIP
from finn.transformation.fpgadataflow.insert_dwc import InsertDWC
from finn.transformation.fpgadataflow.insert_fifo import InsertFIFO
from finn.transformation.fpgadataflow.make_zynq_proj import ZynqBuild

# --- 1. CONFIGURATION & PATHS ---
BUILD_DIR   = pathlib.Path('/home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/')
BOARD       = "Pynq-Z2"
FPGA_PART   = "xc7z020clg400-1"
CLK_NS      = 20.0 

FOLDED      = str(BUILD_DIR / "fashion_cnn_folded.onnx")
SYNTH_MODEL = str(BUILD_DIR / "fashion_cnn_hlssynth.onnx")
ZYNQ_MODEL  = str(BUILD_DIR / "fashion_cnn_zynq.onnx")

# --- 2. HLS SYNTHESIS ---
print(f"📂 Loading folded model from: {FOLDED}")
df_model = ModelWrapper(FOLDED)
df_model = df_model.transform(GiveUniqueNodeNames())

print("⚙️ PrepareIP …")
df_model = df_model.transform(PrepareIP(FPGA_PART, CLK_NS))

print("⚙️ HLSSynthIP … (Expect 10–20 min)")
df_model = df_model.transform(HLSSynthIP())
df_model.save(SYNTH_MODEL)
print(f"✅ HLS done → {SYNTH_MODEL}")

# --- 3. DATA HIGHWAY OPTIMIZATION ---
synth_model = ModelWrapper(SYNTH_MODEL)
print("\n🛠️ Fixing data highway widths (Inserting DWCs)...")
synth_model = synth_model.transform(InsertDWC())

print("💉 Injecting FIFOs for hardware stability...")
synth_model = synth_model.transform(InsertFIFO(True))

# 🌟 THE TRUE FIX: Force all FIFOs and Line Buffers into idle LUTs!
print("🗜️ Forcing FIFOs and Sliding Windows out of BRAM and into LUTs...")
for node in synth_model.graph.node:
    if node.op_type == "StreamingFIFO":
        inst = getCustomOp(node)
        inst.set_nodeattr("ram_style", "distributed")
    elif node.op_type.startswith("ConvolutionInputGenerator"):
        inst = getCustomOp(node)
        inst.set_nodeattr("ram_style", "distributed")

synth_model = synth_model.transform(GiveUniqueNodeNames())

# --- 4. THE VIVADO BUILD ---
print(f"\n🚀 Launching ZynqBuild for {BOARD}...")
print("Vivado Synthesis & Implementation started. Time for a coffee break! ☕")

synth_model = synth_model.transform(ZynqBuild(platform=BOARD, period_ns=CLK_NS))
synth_model.save(ZYNQ_MODEL)

# --- 5. LOCATE THE PRIZE ---
bitfile_path = synth_model.get_metadata_prop('bitfile')
if bitfile_path:
    print(f"\n✅ SUCCESS! Bitfile generated at: {bitfile_path}")
    hwh_path = bitfile_path.replace(".bit", ".hwh")
    print(f"📦 HWH file located at: {hwh_path}")
else:
    print("\n❌ Build finished but bitfile path not found in metadata. Check Vivado logs.")

📂 Loading folded model from: /home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/fashion_cnn_folded.onnx
⚙️ PrepareIP …
⚙️ HLSSynthIP … (Expect 10–20 min)
✅ HLS done → /home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/fashion_cnn_hlssynth.onnx

🛠️ Fixing data highway widths (Inserting DWCs)...
💉 Injecting FIFOs for hardware stability...
🗜️ Forcing FIFOs and Sliding Windows out of BRAM and into LUTs...

🚀 Launching ZynqBuild for Pynq-Z2...
Vivado Synthesis & Implementation started. Time for a coffee break! ☕


/home/yeager/finn/src/finn/transformation/fpgadataflow/floorplan.py:107: UserWarning: 55 nodes have no entry in the provided floorplan, SLR was set to -1
  warnings.warn(
/home/yeager/finn/src/finn/transformation/fpgadataflow/prepare_ip.py:56: UserWarning: Using pre-existing code for StreamingDataflowPartition_1_FMPadding_rtl_0
  warnings.warn("Using pre-existing code for %s" % node.name)
/home/yeager/finn/src/finn/transformation/fpgadataflow/prepare_ip.py:56: UserWarning: Using pre-existing code for StreamingDataflowPartition_1_ConvolutionInputGenerator_rtl_0
  warnings.warn("Using pre-existing code for %s" % node.name)
/home/yeager/finn/src/finn/transformation/fpgadataflow/prepare_ip.py:56: UserWarning: Using pre-existing code for StreamingDataflowPartition_1_MVAU_hls_0
  warnings.warn("Using pre-existing code for %s" % node.name)
/home/yeager/finn/src/finn/transformation/fpgadataflow/prepare_ip.py:56: UserWarning: Using pre-existing code for StreamingDataflowPartition_1_StreamingMax

Exception: Synthesis failed, no bitfile found. Check logs under /home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/vivado_output/vivado_zynq_proj_5rjdq4hc

## Cell 10 — Driver + ZIP

In [28]:
import os, pathlib
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.general import GiveUniqueNodeNames
from qonnx.custom_op.registry import getCustomOp
from finn.transformation.fpgadataflow.prepare_ip import PrepareIP
from finn.transformation.fpgadataflow.hlssynth_ip import HLSSynthIP
from finn.transformation.fpgadataflow.insert_dwc import InsertDWC
from finn.transformation.fpgadataflow.insert_fifo import InsertFIFO
from finn.transformation.fpgadataflow.make_zynq_proj import ZynqBuild

# --- 1. CONFIGURATION ---
BUILD_DIR   = pathlib.Path('/home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/')
BOARD       = "Pynq-Z2"
FPGA_PART   = "xc7z020clg400-1"
CLK_NS      = 20.0 

FOLDED      = str(BUILD_DIR / "fashion_cnn_folded.onnx")
SYNTH_MODEL = str(BUILD_DIR / "fashion_cnn_hlssynth.onnx")
ZYNQ_MODEL  = str(BUILD_DIR / "fashion_cnn_zynq.onnx")

# --- 2. NUCLEAR CACHE WIPE ---
print("🧹 Executing nuclear wipe of Vivado IP cache...")
os.system(f"rm -rf {BUILD_DIR}/code_gen_ipgen_*")
os.system(f"rm -rf {BUILD_DIR}/vivado_zynq_proj_*")
os.system("rm -rf /tmp/finn_dev_*")

# --- 3. PASS 1: CONSTRAIN COMPUTE & MEMORY ---
print(f"📂 Loading folded model: {FOLDED}")
df_model = ModelWrapper(FOLDED)

print("\n🗜️ PASS 1: Forcing Layers & Sliding Windows into LUTs...")
for node in df_model.graph.node:
    if node.op_type in ["MVAU_hls", "MVAU_rtl"]:
        inst = getCustomOp(node)
        # ONLY let the final massive classifier use BRAM
        if node.name.startswith("MVAU_hls_4") or node.name.startswith("MVAU_rtl_4"):
            inst.set_nodeattr("ram_style", "block")
            print(f"  -> {node.name:40} : Locked to BRAM")
        else:
            inst.set_nodeattr("ram_style", "distributed")
            print(f"  -> {node.name:40} : Forced to LUTs")
            
    elif node.op_type in ["ConvolutionInputGenerator_hls", "ConvolutionInputGenerator_rtl"]:
        inst = getCustomOp(node)
        inst.set_nodeattr("ram_style", "distributed")
        print(f"  -> {node.name:40} : Forced to LUTs")

df_model = df_model.transform(GiveUniqueNodeNames())

print("\n⚙️ PrepareIP …")
df_model = df_model.transform(PrepareIP(FPGA_PART, CLK_NS))

print("⚙️ HLSSynthIP … (Rebuilding fresh IPs, ~15 mins)")
df_model = df_model.transform(HLSSynthIP())
df_model.save(SYNTH_MODEL)
print(f"✅ HLS done → {SYNTH_MODEL}")

# --- 4. PASS 2: CONSTRAIN INJECTED FIFOS ---
synth_model = ModelWrapper(SYNTH_MODEL)
print("\n🛠️ Inserting Data Width Converters and FIFOs...")
synth_model = synth_model.transform(InsertDWC())
synth_model = synth_model.transform(InsertFIFO(True))

print("\n🗜️ PASS 2: Forcing newly injected FIFOs into LUTs...")
for node in synth_model.graph.node:
    if node.op_type == "StreamingFIFO":
        inst = getCustomOp(node)
        inst.set_nodeattr("ram_style", "distributed")
        print(f"  -> {node.name:40} : Forced to LUTs")

synth_model = synth_model.transform(GiveUniqueNodeNames())

# --- 5. THE VIVADO BUILD ---
print(f"\n🚀 Launching ZynqBuild for {BOARD}...")
print("Vivado Synthesis & Implementation started. Time for a well-deserved coffee! ☕")
synth_model = synth_model.transform(ZynqBuild(platform=BOARD, period_ns=CLK_NS))
synth_model.save(ZYNQ_MODEL)

# --- 6. LOCATE THE PRIZE ---
bitfile_path = synth_model.get_metadata_prop('bitfile')
if bitfile_path:
    print(f"\n✅ SUCCESS! Bitfile generated at: {bitfile_path}")
    hwh_path = bitfile_path.replace(".bit", ".hwh")
    print(f"📦 HWH file located at: {hwh_path}")
else:
    print("\n❌ Build finished but bitfile path not found in metadata.")

🧹 Executing nuclear wipe of Vivado IP cache...
📂 Loading folded model: /home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/fashion_cnn_folded.onnx

🗜️ PASS 1: Forcing Layers & Sliding Windows into LUTs...
  -> ConvolutionInputGenerator_rtl_0          : Forced to LUTs
  -> MVAU_hls_0                               : Forced to LUTs
  -> ConvolutionInputGenerator_rtl_1          : Forced to LUTs
  -> MVAU_hls_1                               : Forced to LUTs
  -> ConvolutionInputGenerator_rtl_2          : Forced to LUTs
  -> MVAU_hls_2                               : Forced to LUTs
  -> ConvolutionInputGenerator_rtl_3          : Forced to LUTs
  -> MVAU_hls_3                               : Forced to LUTs
  -> MVAU_hls_4                               : Locked to BRAM

⚙️ PrepareIP …
⚙️ HLSSynthIP … (Rebuilding fresh IPs, ~15 mins)
✅ HLS done → /home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/fashion_cnn_hlssynth.onnx

🛠️ Inserting Data Width Converters and FIFOs...

🗜️ PASS 2:

/home/yeager/finn/src/finn/transformation/fpgadataflow/floorplan.py:107: UserWarning: 55 nodes have no entry in the provided floorplan, SLR was set to -1
  warnings.warn(
/home/yeager/finn/src/finn/transformation/fpgadataflow/prepare_ip.py:56: UserWarning: Using pre-existing code for StreamingDataflowPartition_1_FMPadding_rtl_0
  warnings.warn("Using pre-existing code for %s" % node.name)
/home/yeager/finn/src/finn/transformation/fpgadataflow/prepare_ip.py:56: UserWarning: Using pre-existing code for StreamingDataflowPartition_1_ConvolutionInputGenerator_rtl_0
  warnings.warn("Using pre-existing code for %s" % node.name)
/home/yeager/finn/src/finn/transformation/fpgadataflow/prepare_ip.py:56: UserWarning: Using pre-existing code for StreamingDataflowPartition_1_MVAU_hls_0
  warnings.warn("Using pre-existing code for %s" % node.name)
/home/yeager/finn/src/finn/transformation/fpgadataflow/prepare_ip.py:56: UserWarning: Using pre-existing code for StreamingDataflowPartition_1_StreamingMax


✅ SUCCESS! Bitfile generated at: /home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/vivado_output/vivado_zynq_proj_evbyscp4/resizer.bit
📦 HWH file located at: /home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/vivado_output/vivado_zynq_proj_evbyscp4/resizer.hwh


In [29]:
import os
import shutil
from qonnx.core.modelwrapper import ModelWrapper
from finn.transformation.fpgadataflow.make_pynq_driver import MakePYNQDriver

# Load your completed Zynq model
ZYNQ_MODEL = '/home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/fashion_cnn_zynq.onnx'
model = ModelWrapper(ZYNQ_MODEL)

print("🐍 Generating PYNQ Python driver...")
# Create the Python wrapper for the board
model = model.transform(MakePYNQDriver(platform="zynq-iodma"))

# Grab the paths to your finished files
driver_dir = model.get_metadata_prop("pynq_driver_dir")
bitfile = model.get_metadata_prop("bitfile")
hwh_file = bitfile.replace(".bit", ".hwh")

# Create a clean deployment folder
deploy_dir = "/home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/deploy"
os.makedirs(deploy_dir, exist_ok=True)

print("📦 Assembling deployment package...")
# Move everything into the folder
shutil.copytree(driver_dir, deploy_dir + "/driver", dirs_exist_ok=True)
shutil.copy(bitfile, deploy_dir + "/resizer.bit")
shutil.copy(hwh_file, deploy_dir + "/resizer.hwh")

# Zip it all up
shutil.make_archive(deploy_dir, 'zip', deploy_dir)

print(f"🎉 DONE! Your deployment package is ready to download: {deploy_dir}.zip")

🐍 Generating PYNQ Python driver...
📦 Assembling deployment package...
🎉 DONE! Your deployment package is ready to download: /home/yeager/finn/notebooks/claude_builds/mnist_1_fashion/deploy.zip
